# Fase 5 · Texto, voz y cara de cada frase (Bloque 2)

**Objetivo:** situar cada frase de la transcripción oficial en el vídeo y unir, en la ventana de cada frase, lo que se dice (postura, bloque 2), cómo se dice (tono de voz, bloque 1) y cómo se ve (expresión facial, bloque 3). Con los tiempos, las citas del briefing y del chat saltan al segundo exacto del vídeo, y los momentos clave pueden salir también de la voz y de la cara.

In [1]:
# autoreload solo vigila nuestro código (src), no las librerías instaladas
%load_ext autoreload
%autoreload 1
%aimport src.config, src.text.alignment, src.text.alignment_eval, src.text.fusion

import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

from src.config import ROOT, EVENTS_DIR, HISTORY_DIR
from src.text.alignment_eval import validar

resultados_dir = ROOT / "benchmarks" / "results" / "fusion"
resultados_dir.mkdir(parents=True, exist_ok=True)
corpus = pd.read_parquet(HISTORY_DIR / "transcripts.parquet")
REFERENCIA = "2026-09-10"

## 1. Diseño

- **Tiempo de cada frase.** La transcripción oficial no tiene tiempos; la del audio (`transcript.csv`, bloque 1) sí, pero contiene errores de reconocimiento y no coincide palabra por palabra con la oficial, que está editada. Las dos secuencias de palabras normalizadas se emparejan con el algoritmo de bloques comunes más largos (`difflib.SequenceMatcher`), que respeta el orden y tolera omisiones y sustituciones. Cada frase toma el tiempo de su primera y de su última palabra emparejadas; las palabras sin pareja se interpolan entre sus vecinas. Se alinea la transcripción completa, preguntas incluidas, porque el audio también las contiene. Si el bloque 1 entrega segmentos en lugar de palabras, el tiempo de cada segmento se reparte entre sus palabras en proporción a su longitud.
- **Voz y cara por frase.** Media de cada señal en la ventana de la frase, ponderada por los segundos de solapamiento. La cara solo cuenta en los fotogramas con la presidenta en primer plano: durante las preguntas, la realización enfoca a los periodistas.
- **Desviación, no nivel.** Cada señal se expresa también como desviación respecto a la media del hablante en la rueda, en desviaciones típicas. En una banquera central el nivel dice poco, porque su expresión es casi siempre la misma; lo informativo es el cambio.
- **Momentos clave.** A los cuatro momentos de texto de la fase 6 se añaden hasta dos por cómo se dice: la frase relevante de la presidenta con la voz más activada y la de expresión facial más alejada de su media, si superan 1,5 desviaciones típicas.
- **Resumen de la rueda.** `summary.json` recibe los tiempos de citas y momentos clave, la voz y la cara medias de la presidenta y su posición frente a las demás ruedas procesadas, cuando hay al menos tres. La etiqueta de la cara también es relativa: en la rueda de referencia ninguna emoción supera el 21 % de probabilidad media, así que la emoción con más probabilidad no describe nada.
- **Ejecución.** `scripts/fusion_eventos.py` lo hace para cada rueda con `transcript.csv`. Voz y cara son opcionales: si llegan después, basta con volver a ejecutarlo.

## 2. Validación de la alineación con audio simulado

Antes de disponer de la transcripción real, el error de la alineación se mide en un caso con solución conocida. A partir de la transcripción oficial de la rueda de referencia se simula la salida de un reconocedor de voz: tiempos por palabra a unas 2,4 palabras por segundo, palabras omitidas y mal reconocidas, muletillas y segmentos de 8 a 20 palabras, como los de Whisper. Se prueban cuatro niveles de error, con cinco simulaciones cada uno.

**Requisito.** El uso principal de los tiempos es saltar al inicio de una frase desde una cita o un momento clave, y un desfase de 1 a 2 segundos apenas se percibe. Se exige un error mediano inferior a 1 s y un percentil 95 inferior a 3 s con un 15 % de palabras mal reconocidas.

In [2]:
validacion = validar(corpus[corpus["date"] == REFERENCIA], niveles=[0.0, 0.08, 0.15, 0.25])
validacion.to_csv(resultados_dir / "validacion_alineacion.csv", index=False)
validacion

,error_palabras,mediana_s,p95_s,max_s,menos_de_1s,cobertura_media
0,0.00,0.22,1.03,3.30,0.945,1.000
1,0.08,0.28,1.16,3.29,0.932,0.919
2,0.15,0.28,1.31,3.85,0.919,0.852
3,0.25,0.33,1.32,15.42,0.894,0.754


## 3. Ruedas con transcripción del audio

`scripts/fusion_eventos.py` se aplica a todas las ruedas que ya tienen `transcript.csv`. Para cada una informa de la proporción de frases del panel con al menos la mitad de sus palabras encontradas en el audio, de si el orden temporal es correcto y de qué proporción de frases tiene voz y cara.

In [3]:
salida = subprocess.run([sys.executable, "scripts/fusion_eventos.py"], cwd=ROOT, capture_output=True, text=True)
print(salida.stdout or salida.stderr)

Ninguna rueda tiene todavía transcript.csv (bloque 1)



## 4. Señales de la rueda de referencia

Postura de cada frase, tono de voz y expresión facial a lo largo de la rueda; las líneas de puntos marcan los momentos clave. Las señales que aún no se han entregado aparecen como pendientes.

In [4]:
ruta = EVENTS_DIR / REFERENCIA / "signals.csv"
senales = pd.read_csv(ruta) if ruta.exists() else None
if senales is None or "voice_arousal_z" not in senales or senales["start"].isna().all():
    print("Pendiente: la rueda de referencia aún no tiene transcript.csv del bloque 1")
else:
    resumen = json.loads((EVENTS_DIR / REFERENCIA / "summary.json").read_text(encoding="utf-8"))
    # Paleta del proyecto: naranja = hawkish (como las subidas), aguamarina = dovish (como las bajadas)
    NARANJA, AGUAMARINA, GRIS, AZUL = "#eb6834", "#1baf7a", "#a3a29d", "#2a78d6"
    TINTA, TINTA_SECUNDARIA, REJILLA = "#0b0b0b", "#52514e", "#e6e5e0"

    minutos = senales["start"] / 60
    fig, ejes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    ejes[0].bar(minutos, senales["score"], width=(senales["end"] - senales["start"]) / 60, align="edge",
                color=senales["label"].map({"hawkish": NARANJA, "dovish": AGUAMARINA, "neutral": GRIS}))
    ejes[0].set_ylim(-1.05, 1.05)
    ejes[0].set_ylabel("Postura", color=TINTA_SECUNDARIA)
    ejes[0].legend(handles=[Patch(color=NARANJA, label="Hawkish"), Patch(color=GRIS, label="Neutral"),
                            Patch(color=AGUAMARINA, label="Dovish")], frameon=False, ncol=3,
                   loc="lower right", bbox_to_anchor=(1, 1))
    for eje, columna, titulo in [(ejes[1], "voice_arousal_z", "Voz: activación (z)"),
                                 (ejes[2], "face_valence_z", "Cara: valencia (z)")]:
        if senales[columna].notna().any():
            eje.plot(minutos, senales[columna], color=AZUL, linewidth=1.2, marker="o", markersize=2)
        else:
            eje.text(0.5, 0.5, "Pendiente de entrega", transform=eje.transAxes, ha="center", va="center",
                     color=TINTA_SECUNDARIA)
        eje.set_ylabel(titulo, color=TINTA_SECUNDARIA)
    for eje in ejes:
        eje.axhline(0, color=TINTA_SECUNDARIA, linewidth=0.8)
        eje.grid(axis="y", color=REJILLA, linewidth=0.8)
        eje.spines[["top", "right"]].set_visible(False)
        eje.tick_params(colors=TINTA_SECUNDARIA)
        for momento in resumen["key_moments"]:
            if momento["start"] is not None:
                eje.axvline(momento["start"] / 60, color=TINTA, linestyle=":", linewidth=0.8)
    ejes[2].set_xlabel("Minuto del vídeo", color=TINTA_SECUNDARIA)
    ejes[0].set_title(f"Texto, voz y cara en la rueda de prensa del {REFERENCIA}", loc="left", color=TINTA)
    fig.tight_layout()
    fig.savefig(resultados_dir / f"senales_{REFERENCIA}.png", dpi=150)
    plt.show()

    for momento in resumen["key_moments"]:
        print(f"[{momento['start'] / 60 if momento['start'] is not None else float('nan'):5.1f} min] "
              f"{momento['reason']}: {momento['text']}")

Pendiente: la rueda de referencia aún no tiene transcript.csv del bloque 1


## 5. Decisión

Se redacta a la vista de los resultados.